# 06.03 · Prompt templates & versioning

**Chapter:** Prompt & Context Engineering · **Milestone:** M2

## Goals
- [ ] Template options: f-strings, `string.Template`, Jinja2 — and their pitfalls
- [ ] Store prompts as versioned files, load them at runtime, log the version with every call
- [ ] Know prompt registries (Langfuse, LangSmith, PromptLayer) and when they're worth it

## Resources
- [Jinja2 template designer docs](https://jinja.palletsprojects.com/en/stable/templates/) — free
- [Langfuse – Prompt management](https://langfuse.com/docs/prompts) · [LangSmith – Prompt hub](https://docs.smith.langchain.com/prompt_engineering) — free tiers

Prompts are code: they change behaviour, need review, need rollback. A prompt buried in an f-string inside a function is code
you can't diff, test or swap without a deploy.

In [ ]:
import json
import string
from pathlib import Path

from jinja2 import Environment, FileSystemLoader, StrictUndefined

from aieng.llm import client

## 1. f-strings: fine for tiny prompts, fragile otherwise
Pitfall: literal braces (JSON examples!) must be doubled, and there are no loops/conditionals.

In [ ]:
ticket = "I was charged twice."
prompt = f"Classify: {ticket}\nReturn JSON like {{\"category\": \"...\"}}"  # note the {{ }}
print(prompt)

## 2. `string.Template`: safe substitution, `$name` syntax
Braces are left alone, so JSON examples are easy. `safe_substitute` won't crash on missing keys (which can also hide bugs).

In [ ]:
t = string.Template('Classify: $ticket\nReturn JSON like {"category": "..."}')
print(t.substitute(ticket=ticket))

## 3. Jinja2: loops, conditionals, filters
The standard for real prompt templates (Hugging Face chat templates are Jinja too, 04.04). Use `StrictUndefined` so a missing
variable is an error instead of silently rendering an empty string.

In [ ]:
env = Environment(loader=FileSystemLoader("prompts"), undefined=StrictUndefined, trim_blocks=True, lstrip_blocks=True)
print(Path("prompts/classify_ticket/v2.j2").read_text())

In [ ]:
DEFINITIONS = {
    "billing": "charges, invoices, refunds, payment methods, pricing",
    "bug": "something in the product is broken",
    "feature_request": "asking for new functionality",
    "account": "login, password, email, deletion, ownership",
    "other": "anything else",
}
EXAMPLES = [{"text": "Why did my bill go up?", "label": "billing"}, {"text": "Please add a Zapier integration.", "label": "feature_request"}]

rendered = env.get_template("classify_ticket/v2.j2").render(definitions=DEFINITIONS, examples=EXAMPLES, ticket=ticket)
print(rendered)

try:
    env.get_template("classify_ticket/v2.j2").render(definitions=DEFINITIONS, ticket=ticket)  # forgot examples
except Exception as e:
    print("StrictUndefined caught it:", e)

## 4. Versioned prompt files
Layout: `prompts/<name>/v<N>.j2`. Never edit a released version — add a new file. Load "latest" or pin a version via config,
and **log the version with every call** so you can attribute quality changes.

In [ ]:
def load_prompt(name, version=None):
    versions = sorted(Path("prompts", name).glob("v*.j2"), key=lambda p: int(p.stem[1:]))
    path = versions[-1] if version is None else Path("prompts", name, f"v{version}.j2")
    return env.get_template(f"{name}/{path.name}"), f"{name}@{path.stem}"


c, MODEL = client()
EXTRA = {"reasoning_effort": "none"} if c.base_url.host == "localhost" else {}
tickets = json.loads(Path("tickets.json").read_text())
CATEGORIES = list(DEFINITIONS)


def classify(text, version=None):
    template, prompt_id = load_prompt("classify_ticket", version)
    content = template.render(categories=CATEGORIES, definitions=DEFINITIONS, examples=EXAMPLES, ticket=text)
    out = c.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": content}], temperature=0, **EXTRA)
    answer = out.choices[0].message.content.strip().lower()
    return answer, {"prompt": prompt_id, "model": MODEL, "in_tokens": out.usage.prompt_tokens}  # log this with the call


for v in [1, 2]:
    correct = 0
    for t in tickets:
        answer, meta = classify(t["text"], version=v)
        correct += t["label"] in answer and sum(cat in answer for cat in CATEGORIES) == 1
    print(f"{meta['prompt']:22} accuracy {correct / len(tickets):.0%} | ~{meta['in_tokens']} input tokens")

## 5. Prompt registries
| Approach | Pros | Cons |
|---|---|---|
| **Files in git** (above) | code review, diffs, CI tests, free | changing a prompt = deploy |
| **Registry** (Langfuse, LangSmith, PromptLayer, Braintrust) | non-engineers can edit, instant rollout/rollback, linked to traces & evals | another service; runtime dependency (cache it!) |

Start with files. Move to a registry when product people iterate on prompts daily or you need A/B tests in production (chapter 11).

## Exercises
1. Add `v3.j2` that improves on v2 (e.g. definitions from 06.01's best version). Confirm `load_prompt` picks it up as latest.
2. Write a tiny test: render every template version with sample inputs and assert no `StrictUndefined` error and length < N tokens.
   This is your first "prompt CI" check.
3. Self-host Langfuse with Docker (`docs: langfuse.com/self-hosting`) or use its free cloud tier, move `classify_ticket` there and
   fetch it with the Python SDK.

In [ ]:
# your code here